# AncestryLab — Exact AADR Sample Discovery and Validation

**Project component:** source/target cohort discovery for the AncestryLab ancient-DNA workflow  
**Dataset:** Allen Ancient DNA Resource (AADR), public 1240K release  
**Selection policy:** exact labels and explicit archaeological/site criteria only — **no proxy substitution**.

This notebook prepares a reproducible sample inventory for subsequent population-genetic analyses. It downloads only the compact AADR individual and annotation metadata (`.ind` and `.anno`) needed to identify samples. It does **not** download the multi-gigabyte genotype matrix and does not calculate ancestry proportions.

## Scientific scope and method

1. Query the Harvard Dataverse persistent identifier for the current public AADR release.
2. Download the matching public 1240K `.ind` and `.anno` files, verify file sizes, and record the exact dataset version and file IDs.
3. Join individual records to AADR population-group annotations.
4. Select modern targets only when the AADR group label exactly matches an explicitly listed alias and at least 80% of its annotation rows are marked present-day.
5. Select ancient cohorts only by explicitly declared site/name, political entity, and date-window rules. The Europe and Greece categories are transparently defined composites, not individual archaeological sites.
6. Produce a full audit. Missing requested cohorts remain missing; the notebook never substitutes a nearby, contemporary, genetically similar, or similarly named group.
7. Stop before formal modelling if any required cohort is absent. A missing sample is a research finding, not permission to replace it.

## What this notebook does not claim

The former version contained a simplex-constrained allele-frequency fit and nearest-proxy fallback code. Those are removed. A least-squares mixture is **not** qpAdm, qpWave, qpGraph, LINADMIX, or the formal method of Agranat-Tamir et al. This notebook is therefore an input-validation stage, not a reproduction of that paper's statistical analyses. Formal f-statistics and admixture modelling must be performed with the relevant published method, SNP intersection, outgroups, and model specification, with diagnostics and model-fit statistics reported separately.

The notebook uses public, published AADR genotype metadata only; it does not process an individual's private DNA file.

## Primary resources

- AADR dataset: [Harvard Dataverse DOI 10.7910/DVN/FFIDCW](https://doi.org/10.7910/DVN/FFIDCW)
- AADR resource paper: [Nature Scientific Data (2024)](https://www.nature.com/articles/s41597-024-03031-7)
- Agranat-Tamir et al. (2020), *The Genomic History of the Bronze Age Southern Levant*: [Cell DOI 10.1016/j.cell.2020.04.024](https://doi.org/10.1016/j.cell.2020.04.024)
- Paper supplement: [Harvard Reich Lab PDF](https://reich.hms.harvard.edu/sites/reich.hms.harvard.edu/files/inline-files/2020_AgranatTamir_Cell_Levant_Bronze_Age_Supplement.pdf)
- ADMIXTOOLS: [DReichLab/AdmixTools](https://github.com/DReichLab/AdmixTools)
- ADMIXTOOLS 2: [uqrmaie1/admixtools](https://github.com/uqrmaie1/admixtools)

> **Reproducibility rule:** cite the exact AADR version, filenames, file IDs, selected AADR group IDs, sample counts, date ranges, and source studies in any downstream report. Do not silently rename a cohort or merge groups without documenting the rule.

## 1. Configuration

Edit only the explicitly listed labels and cohort rules below. `STRICT_NO_PROXY` is deliberately fixed to `True`; there is no nearest-neighbour or proxy option in this notebook. The broad regional composites are named as composites so they cannot be mistaken for a single ancient population.

In [ ]:
# Standard-library and scientific Python dependencies.
# In Google Colab, these packages are usually preinstalled; install only if missing.
import importlib.util
import subprocess
import sys

_REQUIRED = {"numpy": "numpy", "pandas": "pandas", "requests": "requests", "tqdm": "tqdm"}
_missing = [package for module, package in _REQUIRED.items()
            if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

from pathlib import Path
import hashlib
import json
import re
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from IPython.display import display

# Harvard Dataverse persistent ID for the public AADR release.
DATASET_PID = "doi:10.7910/DVN/FFIDCW"
DATAVERSE_BASE = "https://dataverse.harvard.edu"

# Colab path; change this to a mounted persistent folder if desired.
WORKDIR = Path("/content/ancestrylab_aadr_exact_samples")
WORKDIR.mkdir(parents=True, exist_ok=True)

# Do not relax this policy: the code below has no proxy-selection implementation.
STRICT_NO_PROXY = True
MIN_PRESENT_DAY_FRACTION = 0.80

# Modern target labels: aliases are exact, case-insensitive AADR group labels.
MODERN_TARGET_ALIASES = {
    "English": ["English"],
    "Moroccan": ["Moroccan", "Morocco"],
    "Spanish": ["Spanish"],
    "Northern Italian": ["Italian_North", "Northern_Italian", "Italian_Northern"],
    "Southern Italian": ["Italian_South", "Southern_Italian", "Italian_Southern"],
    "Sicilian": ["Sicilian", "Sicily_Modern", "Italian_Sicily_Modern"],
    "French": ["French"],
    "Egyptian": ["Egyptian", "Egypt_Modern", "Egypt"],
    "Iranian": ["Iranian", "Iran_Modern", "Iran"],
}

# Date ranges are in years BP (before 1950 CE), inclusive.
EUROPE_NEOLITHIC_BRONZE_BP = (2700, 8500)
GREECE_BRONZE_BP = (3000, 5200)
IRAN_CHALCOLITHIC_BP = (4000, 7500)

# Country/entity names for the broad Europe composite. Greece is intentionally excluded;
# Greece Bronze Age is recorded as its own separately defined composite.
EUROPE_POLITICAL_ENTITIES = {
    "Albania", "Andorra", "Austria", "Belarus", "Belgium", "Bosnia and Herzegovina",
    "Bulgaria", "Croatia", "Cyprus", "Czechia", "Czech Republic", "Denmark", "Estonia",
    "Finland", "France", "Germany", "Hungary", "Iceland", "Ireland", "Italy", "Kosovo",
    "Latvia", "Liechtenstein", "Lithuania", "Luxembourg", "Malta", "Moldova", "Monaco",
    "Montenegro", "Netherlands", "North Macedonia", "Norway", "Poland", "Portugal",
    "Romania", "San Marino", "Serbia", "Slovakia", "Slovenia", "Spain", "Sweden",
    "Switzerland", "Ukraine", "United Kingdom", "England", "Scotland", "Wales",
}

# Reusable HTTP session with explicit identification and conservative timeouts.
http = requests.Session()
http.headers.update({"User-Agent": "AncestryLab-AADR-sample-audit/1.0", "Accept": "application/json"})

assert STRICT_NO_PROXY is True, "The AncestryLab sample-audit notebook must remain no-proxy."
print("Configuration loaded. Strict no-proxy policy is active.")
print("Working directory:", WORKDIR)

## 2. Resolve the current AADR release and download metadata

The Dataverse API is queried at runtime so the audit records the release actually used. Only `.ind` and `.anno` are downloaded here. Genotype files are intentionally excluded from this sample-discovery stage to avoid large unnecessary downloads and to keep sample selection independent of downstream statistical modelling.

In [ ]:
def get_latest_aadr_files(session: requests.Session):
    """Return the current Dataverse version and its public 1240K file metadata."""
    response = session.get(
        f"{DATAVERSE_BASE}/api/datasets/:persistentId/",
        params={"persistentId": DATASET_PID}, timeout=(20, 90)
    )
    response.raise_for_status()
    payload = response.json()
    if payload.get("status") != "OK":
        raise RuntimeError(f"Dataverse returned a non-OK response: {payload}")

    latest = payload["data"]["latestVersion"]
    data_files = [entry["dataFile"] for entry in latest.get("files", [])]

    def choose(extension: str):
        candidates = [f for f in data_files
                      if f.get("filename", "").lower().endswith("." + extension)
                      and "1240k" in f.get("filename", "").lower()
                      and "pub" in f.get("filename", "").lower()]
        if not candidates:
            raise FileNotFoundError(
                f"No public 1240K .{extension} file found in AADR version "
                f"{latest.get('versionNumber')}. Inspect the Dataverse file listing."
            )
        # Prefer an explicitly named patch release when Dataverse provides one.
        patch = [f for f in candidates if "patch" in f.get("filename", "").lower()]
        if patch:
            candidates = patch
        candidates.sort(key=lambda f: int(f.get("filesize") or 0), reverse=True)
        if len(candidates) > 1 and candidates[0].get("filesize") == candidates[1].get("filesize"):
            raise RuntimeError(f"Ambiguous .{extension} candidates: {[f['filename'] for f in candidates]}")
        return candidates[0]

    selected = {ext: choose(ext) for ext in ("ind", "anno")}
    return latest, selected


def download_dataverse_file(session: requests.Session, info: dict, destination: Path) -> Path:
    """Stream one Dataverse file; verify its advertised byte count and keep partials resumable."""
    destination = Path(destination)
    expected = int(info.get("filesize") or 0)
    partial = destination.with_suffix(destination.suffix + ".part")

    if destination.exists() and (not expected or destination.stat().st_size == expected):
        print("Already present:", destination.name)
        return destination

    if expected and shutil.disk_usage(destination.parent).free < expected + 100_000_000:
        raise OSError(f"Insufficient free disk space for {destination.name}.")

    offset = partial.stat().st_size if partial.exists() else 0
    headers = {"Range": f"bytes={offset}-"} if offset else {}
    with session.get(
        f"{DATAVERSE_BASE}/api/access/datafile/{info['id']}",
        headers=headers, stream=True, timeout=(30, 180), allow_redirects=True
    ) as response:
        if offset and response.status_code == 206:
            mode = "ab"
        else:
            # Server did not honour Range: restart rather than append a full response.
            offset, mode = 0, "wb"
            response.raise_for_status()
        with open(partial, mode) as handle:
            for chunk in tqdm(response.iter_content(chunk_size=2 * 1024 * 1024),
                              desc=destination.name, unit="B", unit_scale=True):
                if chunk:
                    handle.write(chunk)

    actual = partial.stat().st_size
    if expected and actual != expected:
        raise IOError(f"Incomplete download for {destination.name}: {actual:,}/{expected:,} bytes. Rerun to resume.")
    partial.replace(destination)
    return destination

latest_version, AADR_FILE_INFO = get_latest_aadr_files(http)
AADR_VERSION = latest_version.get("versionNumber")
print(f"AADR Dataverse version: {AADR_VERSION}")
for extension, info in AADR_FILE_INFO.items():
    print(f"{extension}: {info['filename']} | file ID {info['id']} | {int(info.get('filesize') or 0):,} bytes")

ind_path = download_dataverse_file(http, AADR_FILE_INFO["ind"], WORKDIR / AADR_FILE_INFO["ind"]["filename"])
anno_path = download_dataverse_file(http, AADR_FILE_INFO["anno"], WORKDIR / AADR_FILE_INFO["anno"]["filename"])

## 3. Parse and reconcile AADR metadata

AADR `.ind` provides individual identifiers and group IDs. AADR `.anno` provides group-level descriptive metadata. The join below is deliberately keyed by the exact AADR group ID; it does not infer ancestry, merge genetically similar populations, or replace missing cohorts.

In [ ]:
# Read individual records. AADR .ind is whitespace-delimited and has sample ID, sex, group ID.
individuals = pd.read_csv(
    ind_path, sep=r"\s+", header=None,
    names=["sample_id", "sex", "group_id"], dtype=str, engine="python"
)
individuals["sample_id"] = individuals["sample_id"].str.strip()
individuals["group_id"] = individuals["group_id"].str.strip()
if individuals.empty or individuals["sample_id"].isna().any():
    raise ValueError("The AADR .ind file is empty or contains missing sample IDs.")
if individuals["sample_id"].duplicated().any():
    duplicated = individuals.loc[individuals["sample_id"].duplicated(), "sample_id"].head().tolist()
    raise ValueError(f"Duplicate sample IDs in .ind file (examples: {duplicated}).")

# Read tab-delimited annotations and locate required fields by their documented headers.
annotations = pd.read_csv(anno_path, sep="\t", dtype=str, low_memory=False)
required = ["Group ID", "Political Entity", "Latitude", "Longitude"]
missing_columns = [column for column in required if column not in annotations.columns]
if missing_columns:
    raise ValueError(f"Required AADR .anno columns are missing: {missing_columns}")

date_mean_col = next((c for c in annotations.columns if c.startswith("Date mean in BP")), None)
full_date_col = next((c for c in annotations.columns if c.startswith("Full Date")), None)
if date_mean_col is None or full_date_col is None:
    raise ValueError("AADR annotation must contain 'Date mean in BP' and 'Full Date' columns.")

annotations = annotations.copy()
annotations["group_id"] = annotations["Group ID"].astype(str).str.strip()
annotations["date_bp"] = pd.to_numeric(annotations[date_mean_col], errors="coerce")
annotations["present_day"] = (
    annotations[full_date_col].astype(str).str.strip().str.casefold().eq("present")
    | annotations["date_bp"].eq(0)
)
annotations["entity"] = annotations["Political Entity"].fillna("").astype(str).str.strip()
annotations["latitude"] = pd.to_numeric(annotations["Latitude"], errors="coerce")
annotations["longitude"] = pd.to_numeric(annotations["Longitude"], errors="coerce")

# Count actual individuals from .ind; summarize group annotations without duplicating samples.
individual_counts = (individuals.groupby("group_id", dropna=False)
                     .agg(n_individuals=("sample_id", "nunique")).reset_index())
annotation_summary = (annotations.groupby("group_id", dropna=False)
    .agg(n_annotation_rows=("group_id", "size"),
         median_date_bp=("date_bp", "median"),
         present_day_fraction=("present_day", "mean"),
         political_entity=("entity", lambda x: x.mode().iat[0] if not x.mode().empty else ""),
         median_latitude=("latitude", "median"),
         median_longitude=("longitude", "median"))
    .reset_index())

group_inventory = individual_counts.merge(annotation_summary, on="group_id", how="left", validate="one_to_one")
group_inventory["n_annotation_rows"] = group_inventory["n_annotation_rows"].fillna(0).astype(int)
group_inventory["present_day_fraction"] = group_inventory["present_day_fraction"].fillna(0.0)
group_inventory["political_entity"] = group_inventory["political_entity"].fillna("")
group_inventory["group_id_lc"] = group_inventory["group_id"].astype(str).str.casefold()

# Report groups that appear in one metadata file but not the other; do not silently discard them.
ind_groups = set(individual_counts["group_id"])
anno_groups = set(annotation_summary["group_id"])
missing_annotation_groups = sorted(ind_groups - anno_groups)
annotation_only_groups = sorted(anno_groups - ind_groups)
print(f"Individuals: {len(individuals):,}; annotated groups: {len(group_inventory):,}")
print(f"Groups without annotation rows: {len(missing_annotation_groups):,}")
print(f"Annotation-only groups: {len(annotation_only_groups):,}")

## 4. Exact cohort selection rules

The selectors below are deterministic and transparent. Modern cohorts require an exact AADR group-label alias plus the present-day threshold. Ancient cohorts require a matching site token, political entity, date window, and (where specified) a period token. Composite groups are enumerated individually in the audit and are not presented as a single homogeneous ancient population.

In [ ]:
def exact_modern_group(alias_list, inventory=group_inventory):
    """Find an exact modern AADR group; never fuzzy-match or select a proxy."""
    for alias in alias_list:
        hit = inventory[
            inventory["group_id_lc"].eq(alias.casefold())
            & inventory["present_day_fraction"].ge(MIN_PRESENT_DAY_FRACTION)
        ]
        if not hit.empty:
            return [str(hit.iloc[0]["group_id"])]
    return []


def is_not_outlier_group(group_id: str) -> bool:
    """Exclude AADR outlier subclusters from automatic regional composites."""
    return re.search(r"(?:-|_)o(?:[A-Z0-9_]|$)", str(group_id), flags=re.IGNORECASE) is None


def ancient_site_groups(site_terms, entities, bp_range, period_patterns=None, inventory=group_inventory):
    """Select ancient groups by explicit site/name + entity + date + optional period token."""
    site_pattern = "|".join(re.escape(term.casefold()) for term in site_terms)
    entity_set = {entity.casefold() for entity in entities}
    mask = inventory["group_id_lc"].str.contains(site_pattern, regex=True, na=False)
    mask &= inventory["political_entity"].astype(str).str.casefold().isin(entity_set)
    mask &= inventory["median_date_bp"].between(bp_range[0], bp_range[1], inclusive="both")
    mask &= inventory["group_id"].map(is_not_outlier_group)
    if period_patterns:
        period_pattern = "|".join(f"(?:{pattern})" for pattern in period_patterns)
        mask &= inventory["group_id_lc"].str.contains(period_pattern, regex=True, na=False)
    return sorted(inventory.loc[mask, "group_id"].astype(str).unique().tolist())


# Modern target cohorts: exact label aliases only.
TARGET_GROUPS = {
    label: exact_modern_group(aliases)
    for label, aliases in MODERN_TARGET_ALIASES.items()
}

# Ancient sources. Site and period filters are intentionally explicit and may return no group.
SOURCE_GROUPS = {
    "Yoruba": exact_modern_group(["Yoruba"]),
    "Iron Age Algeria / Khenchela": ancient_site_groups(
        ["Khenchela"], ["Algeria"], (1800, 3800),
        [r"(?:^|[_\s])ia(?:$|[_\s])", r"iron.?age"]
    ),
    "Late Bronze Age Megiddo": ancient_site_groups(
        ["Megiddo"], ["Israel", "Palestine"], (2700, 3800),
        [r"(?:^|[_\s])(?:lba|mlba)(?:$|[_\s])", r"late.?bronze"]
    ),
    "Bronze Age Sirnak / Turkey": ancient_site_groups(
        ["Sirnak", "Şırnak", "Sırnak"], ["Turkey", "Türkiye"], (2500, 5200),
        [r"(?:^|[_\s])(?:ba|eba|mba|lba)(?:$|[_\s])", r"bronze"]
    ),
}

# Iran Chalcolithic is a declared multi-group composite, selected by entity/date/period token.
iran_mask = group_inventory["political_entity"].astype(str).str.casefold().eq("iran")
iran_mask &= group_inventory["median_date_bp"].between(*IRAN_CHALCOLITHIC_BP, inclusive="both")
iran_mask &= group_inventory["group_id_lc"].str.contains(
    r"(?:^|[_\s-])(?:c|chl|chalcolithic)(?:$|[_\s-])", regex=True, na=False
)
iran_mask &= group_inventory["group_id"].map(is_not_outlier_group)
SOURCE_GROUPS["Iran Chalcolithic composite"] = sorted(
    group_inventory.loc[iran_mask, "group_id"].astype(str).unique().tolist()
)

# Broad regional composites are explicitly defined and remain labelled as composites.
entity_lc = group_inventory["political_entity"].astype(str).str.casefold()
europe_entity_lc = {entity.casefold() for entity in EUROPE_POLITICAL_ENTITIES}
europe_mask = entity_lc.isin(europe_entity_lc)
europe_mask &= group_inventory["median_date_bp"].between(*EUROPE_NEOLITHIC_BRONZE_BP, inclusive="both")
europe_mask &= group_inventory["group_id"].map(is_not_outlier_group)
SOURCE_GROUPS["Europe Neolithic–Bronze Age composite"] = sorted(
    group_inventory.loc[europe_mask, "group_id"].astype(str).unique().tolist()
)

greece_mask = entity_lc.eq("greece")
greece_mask &= group_inventory["median_date_bp"].between(*GREECE_BRONZE_BP, inclusive="both")
greece_mask &= group_inventory["group_id"].map(is_not_outlier_group)
SOURCE_GROUPS["Greece Bronze Age composite"] = sorted(
    group_inventory.loc[greece_mask, "group_id"].astype(str).unique().tolist()
)

print("Exact modern target groups:")
for label, group_ids in TARGET_GROUPS.items():
    print(f"  {label}: {group_ids if group_ids else 'MISSING — no proxy will be substituted'}")
print("\nAncient source groups:")
for label, group_ids in SOURCE_GROUPS.items():
    print(f"  {label}: {len(group_ids)} exact-rule group(s)")

## 5. Selection audit and reproducibility exports

The audit includes every requested target and source, whether it was found, the exact AADR group IDs, sample counts, and the selection rule. Missing cohorts are retained as missing rows. No substitute is selected.

In [ ]:
def sample_count_for(group_ids):
    """Count unique individuals in the listed AADR groups."""
    if not group_ids:
        return 0
    return int(group_inventory.loc[group_inventory["group_id"].isin(group_ids), "n_individuals"].sum())


audit_rows = []
for role, selections in (("target", TARGET_GROUPS), ("source", SOURCE_GROUPS)):
    for label, group_ids in selections.items():
        found = bool(group_ids)
        audit_rows.append({
            "role": role,
            "requested_cohort": label,
            "status": "FOUND — exact rule" if found else "MISSING — no proxy permitted",
            "aadr_group_ids": "; ".join(group_ids),
            "n_groups": len(group_ids),
            "n_individuals": sample_count_for(group_ids),
            "selection_policy": "exact modern label + present-day annotation" if role == "target" and label in TARGET_GROUPS
                else "explicit site/entity/date/period rule or named regional composite",
        })

selection_audit = pd.DataFrame(audit_rows)
missing_targets = [label for label, ids in TARGET_GROUPS.items() if not ids]
missing_sources = [label for label, ids in SOURCE_GROUPS.items() if not ids]

# Overlap checks are important before modelling: the same AADR group must not be both
# a target and a source, nor silently appear in two separately interpreted source groups.
source_group_owner = {}
source_overlaps = []
for label, group_ids in SOURCE_GROUPS.items():
    for group_id in group_ids:
        if group_id in source_group_owner:
            source_overlaps.append((group_id, source_group_owner[group_id], label))
        else:
            source_group_owner[group_id] = label

target_group_ids = {gid for ids in TARGET_GROUPS.values() for gid in ids}
source_group_ids = set(source_group_owner)
target_source_overlap = sorted(target_group_ids & source_group_ids)

# Save the complete inventory and audit before deciding whether a downstream model can run.
inventory_csv = WORKDIR / "aadr_group_inventory.csv"
audit_csv = WORKDIR / "exact_sample_selection_audit.csv"
missing_csv = WORKDIR / "missing_requested_cohorts.csv"
provenance_json = WORKDIR / "sample_discovery_provenance.json"

group_inventory.drop(columns=["group_id_lc"]).to_csv(inventory_csv, index=False)
selection_audit.to_csv(audit_csv, index=False)
selection_audit[selection_audit["status"].str.startswith("MISSING")].to_csv(missing_csv, index=False)

provenance = {
    "project": "AncestryLab",
    "component": "AADR exact sample discovery and validation",
    "dataset": "Allen Ancient DNA Resource",
    "dataverse_persistent_id": DATASET_PID,
    "dataverse_version": AADR_VERSION,
    "metadata_files": {
        key: {"filename": value["filename"], "dataverse_file_id": value["id"],
              "filesize_bytes": value.get("filesize")}
        for key, value in AADR_FILE_INFO.items()
    },
    "strict_no_proxy": True,
    "modern_target_aliases": MODERN_TARGET_ALIASES,
    "date_windows_bp": {
        "Europe Neolithic–Bronze Age composite": list(EUROPE_NEOLITHIC_BRONZE_BP),
        "Greece Bronze Age composite": list(GREECE_BRONZE_BP),
        "Iran Chalcolithic composite": list(IRAN_CHALCOLITHIC_BP),
    },
    "selected_targets": TARGET_GROUPS,
    "selected_sources": SOURCE_GROUPS,
    "missing_targets": missing_targets,
    "missing_sources": missing_sources,
    "source_group_overlaps": source_overlaps,
    "target_source_group_overlap": target_source_overlap,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "method_scope": "metadata cohort discovery only; no genotype modelling or ancestry estimates",
    "citations": [
        "https://doi.org/10.7910/DVN/FFIDCW",
        "https://www.nature.com/articles/s41597-024-03031-7",
        "https://doi.org/10.1016/j.cell.2020.04.024",
        "https://github.com/DReichLab/AdmixTools",
        "https://github.com/uqrmaie1/admixtools",
    ],
}
provenance_json.write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding="utf-8")

print("\nSelection audit:")
display(selection_audit)
print("\nSaved outputs:")
for path in (inventory_csv, audit_csv, missing_csv, provenance_json):
    print(" -", path)

if source_overlaps:
    print("\nWARNING: source categories share AADR groups; review before modelling:")
    for item in source_overlaps:
        print(" -", item)
if target_source_overlap:
    print("\nERROR: target/source group overlap; downstream modelling must not proceed:", target_source_overlap)

if missing_targets or missing_sources:
    print("\nSTRICT STOP: required cohorts are missing. No replacement cohorts were selected.")
    print("Missing targets:", missing_targets)
    print("Missing sources:", missing_sources)
    print("The audit files above are the valid output of this run; do not interpret them as ancestry estimates.")
else:
    print("\nAll requested cohorts matched the declared rules. This is still only sample validation.")
    print("Proceed to a separately specified qpWave/qpAdm/f-statistics workflow only after reviewing overlaps, SNP coverage, outgroups, and paper-specific sample definitions.")

## 6. Offline validation and limitations

The checks below exercise the exact-label, present-day, ancient-site, and outlier-selection logic using synthetic metadata. They do not replace a live AADR run. A live run requires network access to Harvard Dataverse and must be performed in Colab or another environment with internet access and sufficient storage for the metadata files.

**Downstream modelling is intentionally not executed here.** The published Bronze Age Southern Levant study's formal methods must be reproduced from the paper and supplement, not approximated with a generic least-squares fit. The AncestryLab modelling stage should separately document the exact target and source sample sets, SNP intersection, missing-data treatment, right/outgroup populations, qpWave rank tests, qpAdm model specification, standard errors, p-values, and sensitivity/rotation tests as applicable.

In [ ]:
# Synthetic unit tests: no network, no AADR download, and no proxy fallback.
synthetic = pd.DataFrame([
    {"group_id": "Moroccan", "group_id_lc": "moroccan", "present_day_fraction": 1.0,
     "political_entity": "Morocco", "median_date_bp": 0.0},
    {"group_id": "Morocco_N", "group_id_lc": "morocco_n", "present_day_fraction": 0.0,
     "political_entity": "Morocco", "median_date_bp": 7000.0},
    {"group_id": "Khenchela_IA", "group_id_lc": "khenchela_ia", "present_day_fraction": 0.0,
     "political_entity": "Algeria", "median_date_bp": 2500.0},
    {"group_id": "Khenchela_o1", "group_id_lc": "khenchela_o1", "present_day_fraction": 0.0,
     "political_entity": "Algeria", "median_date_bp": 2500.0},
    {"group_id": "Greece_BA", "group_id_lc": "greece_ba", "present_day_fraction": 0.0,
     "political_entity": "Greece", "median_date_bp": 4000.0},
], columns=["group_id", "group_id_lc", "present_day_fraction", "political_entity", "median_date_bp"])

assert exact_modern_group(["Moroccan"], synthetic) == ["Moroccan"]
assert exact_modern_group(["Morocco_N"], synthetic) == []  # ancient group is not a modern target
assert ancient_site_groups(["Khenchela"], ["Algeria"], (1800, 3800),
                           [r"(?:^|[_\s])ia(?:$|[_\s])"], synthetic) == ["Khenchela_IA"]
assert "Khenchela_o1" not in ancient_site_groups(["Khenchela"], ["Algeria"], (1800, 3800),
                                                  [r"(?:^|[_\s])ia(?:$|[_\s])"], synthetic)
assert STRICT_NO_PROXY is True
print("PASS: synthetic selection tests (exact modern label, ancient period, outlier exclusion, no-proxy policy).")

## 7. Interpretation checklist before formal analysis

- [ ] Confirm that every AADR group ID corresponds to the intended archaeological cohort or modern target.
- [ ] Read the primary publication and supplement for each ancient cohort; do not rely on group labels alone.
- [ ] Check individual-level dates, site context, contamination/authentication, coverage, and relatedness.
- [ ] Confirm the target and source sets do not overlap and that composite categories are defensible.
- [ ] Define the exact SNP intersection and missing-data rules for the chosen statistic.
- [ ] Reproduce the paper-specific outgroups and model topology before running qpWave/qpAdm/qpGraph.
- [ ] Report all failed models and alternative specifications; do not select only favourable fits.
- [ ] Cite AADR and the original study associated with each selected sample.

**Bottom line:** this notebook makes the sample-selection stage auditable and proxy-free. It does not, by itself, establish ancestry proportions or reproduce the paper's formal statistical results.